# 018 — Problemas de satisfacción de restricciones

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

La solución valida el contrato mínimo sin asumir un valor interno específico.


In [ ]:
result = run_lab("logic", seed=18)
assert result["kind"] == "logic"
assert result["evidence"]
show(result)


## Solución 1 — AC-3

```text
arco A→B (A>B): A=1 no tiene B menor → A := {2,3}
arco B→A (A>B): B=3 no tiene A mayor → B := {1,2}
arco B→C (B=C): B=1 no tiene igual en C={2,3} → B := {2}
arco C→B (B=C): C=3 no tiene igual en B={2} → C := {2}
arco A→B otra vez (B cambió): A=2 no tiene B menor en {2} → A := {3}
punto fijo
```

b) `A={3}, B={2}, C={2}` — dominios unitarios: la propagación sola resolvió el
CSP (A=3, B=2, C=2 satisface A>B y B=C). No hace falta backtracking.


## Solución 2 — Australia

a) **SA**: tiene el grado máximo (5 vecinos). Al inicio todos los dominios miden
3, así que MRV empata y desempata la heurística de grado → SA.

b) SA=rojo elimina rojo de WA, NT, Q, NSW, V: todos quedan con {verde, azul};
T conserva {rojo, verde, azul}.

c) Los vecinos de SA forman la cadena WA-NT-Q-NSW-V con 2 colores: basta
alternar. Solución: SA=rojo, WA=verde, NT=azul, Q=verde, NSW=azul, V=verde,
T=cualquiera.


In [ ]:
dominios = {v: {"rojo", "verde", "azul"} for v in ["WA", "NT", "SA", "Q", "NSW", "V", "T"]}
dominios["SA"] = {"rojo"}
for vecino in ["WA", "NT", "Q", "NSW", "V"]:
    dominios[vecino] = dominios[vecino] - {"rojo"}
print({k: sorted(v) for k, v in dominios.items()})
solucion = {"SA": "rojo", "WA": "verde", "NT": "azul", "Q": "verde",
            "NSW": "azul", "V": "verde", "T": "rojo"}
adyacencias = [("SA", x) for x in ["WA", "NT", "Q", "NSW", "V"]] + [
    ("WA", "NT"), ("NT", "Q"), ("Q", "NSW"), ("NSW", "V")]
assert all(solucion[a] != solucion[b] for a, b in adyacencias)
print("solución válida ✔")


## Solución 3 — El ahorro

a) `3^7 = 2 187` asignaciones completas por fuerza bruta.

b) Tras SA, cada vecino queda con **2 colores** y el problema restante es una
cadena de 5 variables binarias (más T libre): se resuelve linealmente
alternando colores, sin retroceso. Fijar la variable correcta primero convirtió
un problema exponencial en uno lineal — ese es el argumento cuantitativo a
favor de MRV/grado.


## Solución 4 — Analogía con el punto fijo

(a) La cola de arcos pendientes equivale al bucle `while changed` del motor:
mientras alguna regla nueva pueda disparar, se sigue revisando. (b) El punto
fijo es el ciclo en que ninguna regla añade hechos (`rules_fired` deja de
crecer). (c) Garantiza que se derivaron **todas** las consecuencias forzadas de
los hechos y reglas dados — como AC-3 garantiza consistencia de arco — pero no
que exista una 'solución' global ni que los hechos iniciales sean correctos:
la consistencia local no implica satisfacibilidad global.


In [ ]:
result = run_lab("logic", seed=18)
assert len(result["result"]["rules_fired"]) == 3
print("punto fijo con 3 reglas disparadas ✔")


## Reflexión

1. En el mini-CSP X<Y<Z, AC-3 resolvió sin buscar. Construye un CSP de 3 variables donde AC-3 deje dominios de tamaño >1 y aún así no exista solución: ¿qué demuestra eso sobre la consistencia de arco?
2. ¿Por qué MRV ('fail first') elige la variable MÁS restringida pero la heurística de valor elige el MENOS restrictivo? ¿Qué optimiza cada una?
3. El laboratorio encadena reglas hasta un punto fijo, igual que AC-3 revisa arcos hasta un punto fijo. ¿Qué comparten ambos algoritmos y en qué se diferencia lo que garantizan?
